## Cell 1: Setup
**What this demonstrates**: Install `rank-bm25`, load API keys from `.env`, and initialise the Anthropic LLM client and OpenAI embeddings — the two external dependencies of the Hybrid RAG pipeline.
**Expected output**: ✓ Setup complete with model names, retriever parameters, and masked key suffixes.

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────────
# rank-bm25: pure-Python BM25Okapi; the rest is the standard RAG stack
%pip install -q langchain langchain-openai langchain-community chromadb python-dotenv rank-bm25

# ── Standard library ─────────────────────────────────────────────────────────
import os
import time
import pathlib
from typing import Any

# ── Third-party ──────────────────────────────────────────────────────────────
from dotenv import load_dotenv, find_dotenv
from anthropic import Anthropic
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi

# ── Load API keys ─────────────────────────────────────────────────────────────
load_dotenv(find_dotenv())
_anthropic_key = os.environ.get('ANTHROPIC_API_KEY', '')
_openai_key    = os.environ.get('OPENAI_API_KEY', '')
assert _anthropic_key, 'ANTHROPIC_API_KEY not set — add it to .env'
assert _openai_key,    'OPENAI_API_KEY not set — add it to .env'

# ── Constants ─────────────────────────────────────────────────────────────────
LLM_MODEL   = 'claude-sonnet-4-6'
EMBED_MODEL = 'text-embedding-3-small'
CHROMA_DIR  = './chroma_hybrid_rag'
TOP_N       = 20   # candidates per retriever before RRF fusion
TOP_K       = 5    # final chunks returned after fusion

# ── Client initialisation ─────────────────────────────────────────────────────
client:     Anthropic        = Anthropic()            # reads ANTHROPIC_API_KEY from env
embeddings: OpenAIEmbeddings = OpenAIEmbeddings(model=EMBED_MODEL)

print('✓ Setup complete')
print(f'  LLM model:       {LLM_MODEL}')
print(f'  Embedding model: {EMBED_MODEL}')
print(f'  Anthropic key:   ...{_anthropic_key[-4:]}')
print(f'  OpenAI key:      ...{_openai_key[-4:]}')
print(f'  Retrievers:      BM25 top-{TOP_N} + Dense top-{TOP_N} → RRF → top-{TOP_K}')

## Cell 2: Data — ISDA Master Agreement Excerpt
**What this demonstrates**: Load the synthetic ISDA agreement, chunk it, and explain why this document is the canonical test for hybrid retrieval — it mixes exact legal identifiers with dense conceptual prose.
**Expected output**: Document name, size, first 200 characters, and chunk count.

In [ ]:
# ── Locate the ISDA document ──────────────────────────────────────────────────
_candidates = [
    pathlib.Path('../../shared/sample_data/isda_excerpt.txt'),
    pathlib.Path('shared/sample_data/isda_excerpt.txt'),
]
SAMPLE_DATA = next((p for p in _candidates if p.exists()), None)
assert SAMPLE_DATA is not None, (
    'Cannot find isda_excerpt.txt. '
    'Run from the repo root or from modules/03_hybrid_rag/.'
)

document_text: str = SAMPLE_DATA.read_text(encoding='utf-8')

# ── Chunk the document ───────────────────────────────────────────────────────
# 400-char chunks with 60-char overlap — smaller than Naive RAG because ISDA
# sections are short and dense; oversized chunks dilute exact-term BM25 scores
splitter = RecursiveCharacterTextSplitter(
    separators=['\n\n', '\n', '. ', ' ', ''],
    chunk_size=400,
    chunk_overlap=60,
)
chunks: list[str] = splitter.split_text(document_text)

print(f'Document:  {SAMPLE_DATA.name}')
print(f'Size:      {len(document_text):,} characters  |  ~{len(document_text.split()):,} words')
print(f'Chunks:    {len(chunks)} (size=400 chars, overlap=60 chars)')
print()
print('First 200 characters:')
print('-' * 52)
print(document_text[:200])
print('-' * 52)
print()
print('Why this document?')
print('  ISDA agreements combine two query types that challenge dense-only retrieval:')
print('  1. Exact legal identifiers: "Section 5(a)(i)", "Minimum Transfer Amount", "MTA"')
print('     → BM25 excels: token match is exact and unambiguous')
print('  2. Conceptual questions: "what triggers a collateral obligation?"')
print('     → Dense excels: semantic proximity across paraphrased clauses')
print('  Hybrid RAG is the only retriever that handles both in the same query.')

## Cell 3: Core — BM25, Dense, and RRF Fusion
**What this demonstrates**: Build both indexes, define the three retrieval functions (`bm25_retrieve`, `dense_retrieve`, `rrf_fuse`), and compose them into a single `hybrid_retrieve` function.
**Expected output**: Index build confirmation and a summary of each function's signature.

In [ ]:
# ── Tokenizer for BM25 ────────────────────────────────────────────────────────
# Simple whitespace+lowercase split — consistent between index time and query time.
# Domain note: do NOT use a generic stopword list here; terms like 'in', 'of', 'under'
# carry legal meaning in clause references (e.g. 'failure under Section 2(a)(i)').
def tokenize(text: str) -> list[str]:
    return text.lower().split()


# ── Build BM25 index ──────────────────────────────────────────────────────────
# BM25Okapi is built in-memory at startup; the full corpus must be available upfront.
# This is an intentional design constraint — if the corpus changes, rebuild the index.
tokenized_corpus: list[list[str]] = [tokenize(chunk) for chunk in chunks]
bm25: BM25Okapi = BM25Okapi(tokenized_corpus)
avg_tokens = sum(len(t) for t in tokenized_corpus) / len(tokenized_corpus)


# ── Build dense (Chroma) index ────────────────────────────────────────────────
# Store chunk_idx in metadata — this is the bridge between the two index spaces.
# Without it, there is no way to look up a dense result's position in the BM25 corpus.
docs_with_meta: list[Document] = [
    Document(page_content=chunk, metadata={'chunk_idx': i})
    for i, chunk in enumerate(chunks)
]
vectorstore = Chroma.from_documents(
    documents=docs_with_meta,
    embedding=embeddings,
    collection_name='hybrid_rag_isda',
    persist_directory=CHROMA_DIR,
    collection_metadata={'hnsw:space': 'cosine'},  # cosine distance in [0, 2]
)

print(f'BM25 index:   {len(chunks)} docs  |  avg {avg_tokens:.1f} tokens/doc')
print(f'Dense index:  {vectorstore._collection.count()} vectors in Chroma')
print()


# ── BM25 retriever ────────────────────────────────────────────────────────────
def bm25_retrieve(query: str, n: int = TOP_N) -> list[tuple[int, float]]:
    """Retrieve top-n chunks by BM25 Okapi score.

    Args:
        query: Natural language query; tokenised identically to the corpus.
        n: Number of results to return.

    Returns:
        List of (chunk_idx, bm25_score) sorted by descending score.
        Zero-score chunks (no token overlap) are excluded.
    """
    scores = bm25.get_scores(tokenize(query))
    # Sort indices by descending score, take top-n, drop zero-overlap results
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:n]
    return [(i, float(scores[i])) for i in top_indices if scores[i] > 0]


# ── Dense retriever ────────────────────────────────────────────────────────────
def dense_retrieve(query: str, n: int = TOP_N) -> list[tuple[int, float]]:
    """Retrieve top-n chunks by cosine similarity.

    Args:
        query: Natural language query; embedded with the same model as the corpus.
        n: Number of results to return.

    Returns:
        List of (chunk_idx, similarity_score) sorted by descending score.
        similarity_score is in [0, 1]; 1.0 = perfect cosine match.
    """
    # similarity_search_with_score returns (Document, cosine_distance); distance ∈ [0, 2]
    # With hnsw:space=cosine: similarity = 1 - (distance / 2) — rescale to [0, 1]
    raw = vectorstore.similarity_search_with_score(query, k=n)
    return [
        (int(doc.metadata['chunk_idx']), 1.0 - dist / 2.0)
        for doc, dist in raw
    ]


# ── RRF fusion ────────────────────────────────────────────────────────────────
def rrf_fuse(
    bm25_ranked:  list[tuple[int, float]],
    dense_ranked: list[tuple[int, float]],
    k:     int = 60,
    top_k: int = TOP_K,
) -> list[tuple[int, float]]:
    """Merge two ranked lists via Reciprocal Rank Fusion.

    RRF score for document d = Σ_i  1 / (k + rank_i(d))
    Rank is 1-indexed: position 1 earns 1/(k+1), position 2 earns 1/(k+2).
    k=60 is the original Ma et al. default; lower k amplifies top-rank advantage.

    A chunk appearing in BOTH lists scores from two contributions — it is
    rewarded for being strong in at least one modality and present in both.
    A chunk absent from a list contributes nothing (not penalised).

    Args:
        bm25_ranked:  (chunk_idx, score) from BM25, sorted descending.
        dense_ranked: (chunk_idx, score) from dense, sorted descending.
        k:     RRF smoothing constant. Default 60 (Ma et al., 2023).
        top_k: Number of results to return after fusion.

    Returns:
        List of (chunk_idx, rrf_score) sorted by descending RRF score, len ≤ top_k.
    """
    rrf_scores: dict[int, float] = {}
    for rank, (chunk_idx, _) in enumerate(bm25_ranked, start=1):
        rrf_scores[chunk_idx] = rrf_scores.get(chunk_idx, 0.0) + 1.0 / (k + rank)
    for rank, (chunk_idx, _) in enumerate(dense_ranked, start=1):
        rrf_scores[chunk_idx] = rrf_scores.get(chunk_idx, 0.0) + 1.0 / (k + rank)
    sorted_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_results[:top_k]


# ── Hybrid retrieval pipeline ─────────────────────────────────────────────────
def hybrid_retrieve(
    query: str,
    n:     int = TOP_N,
    top_k: int = TOP_K,
) -> list[dict[str, Any]]:
    """Run BM25 + dense retrieval, fuse via RRF, return enriched top-k results.

    Args:
        query: User query string.
        n:     Candidates per retriever before fusion.
        top_k: Final results to return after fusion.

    Returns:
        List of dicts: chunk_idx, text, rrf_score, in_bm25, in_dense.
        in_bm25 / in_dense flags record which retriever(s) found each chunk.
    """
    bm25_res  = bm25_retrieve(query, n)
    dense_res = dense_retrieve(query, n)
    fused     = rrf_fuse(bm25_res, dense_res, top_k=top_k)

    bm25_set  = {idx for idx, _ in bm25_res}
    dense_set = {idx for idx, _ in dense_res}

    return [
        {
            'chunk_idx': idx,
            'text':      chunks[idx],
            'rrf_score': score,
            'in_bm25':   idx in bm25_set,
            'in_dense':  idx in dense_set,
        }
        for idx, score in fused
    ]


print('Retrieval functions defined:')
print('  bm25_retrieve(query, n)                  → list[(chunk_idx, bm25_score)]')
print('  dense_retrieve(query, n)                 → list[(chunk_idx, similarity_score)]')
print('  rrf_fuse(bm25_ranked, dense_ranked, k)   → list[(chunk_idx, rrf_score)]')
print('  hybrid_retrieve(query, n, top_k)         → list[dict]  (in_bm25 + in_dense flags)')

## Cell 4: Run — End-to-End Hybrid Retrieval
**What this demonstrates**: The full Hybrid RAG pipeline on the canonical ISDA margin call query — a query that requires both exact phrase matching and semantic context retrieval.
**Expected output**: Top-5 hybrid chunks with RRF scores and source flags (BM25 / Dense / Both), followed by a grounded answer with section citations and latency breakdown.

In [ ]:
QUERY = "Find all references to 'margin call' triggers in the ISDA agreement"

# ── Step 1: Hybrid retrieval ──────────────────────────────────────────────────
t0 = time.perf_counter()
hybrid_results: list[dict[str, Any]] = hybrid_retrieve(QUERY)
retrieval_ms = (time.perf_counter() - t0) * 1000

# ── Step 2: Assemble context with provenance labels ───────────────────────────
# Label each chunk with its source (BM25/Dense/Both) — the LLM sees the provenance.
# This is informational only; the LLM is instructed to answer from content, not labels.
context_parts = []
for i, r in enumerate(hybrid_results):
    src = 'BM25+Dense' if r['in_bm25'] and r['in_dense'] else 'BM25 only' if r['in_bm25'] else 'Dense only'
    context_parts.append(
        f"[Chunk {i+1} | RRF={r['rrf_score']:.5f} | source={src}]\n{r['text']}"
    )
context = '\n\n---\n\n'.join(context_parts)

SYSTEM = (
    'You are a financial contracts analyst specialising in ISDA agreements. '
    'Answer using ONLY the provided document excerpts. '
    'Cite the specific Section (e.g. Section 1A(a)(ii)) for every claim. '
    'Highlight any numeric thresholds (MTA, amounts, timeframes) verbatim. '
    'If the context is insufficient, say so explicitly.'
)

# ── Step 3: Generate ──────────────────────────────────────────────────────────
t1 = time.perf_counter()
response = client.messages.create(
    model=LLM_MODEL,
    max_tokens=512,
    system=SYSTEM,
    messages=[{
        'role': 'user',
        'content': f'ISDA agreement excerpts:\n{context}\n\nQuestion: {QUERY}'
    }],
)
answer: str = response.content[0].text
generation_ms = (time.perf_counter() - t1) * 1000

# ── Print results ──────────────────────────────────────────────────────────────
print(f'Query: {QUERY!r}')
print()
print(f'Hybrid top-{len(hybrid_results)} chunks (BM25 top-{TOP_N} + Dense top-{TOP_N} → RRF k=60):')
for i, r in enumerate(hybrid_results, 1):
    src = 'BM25+Dense' if r['in_bm25'] and r['in_dense'] else 'BM25 only ' if r['in_bm25'] else 'Dense only'
    preview = r['text'][:80].replace('\n', ' ')
    print(f'  [{i}] RRF={r["rrf_score"]:.5f}  [{src}]  {preview!r}...')
print()
print('=' * 65)
print('ANSWER')
print('=' * 65)
print(answer)
print('=' * 65)
print()
print(f'Retrieval: {retrieval_ms:.0f} ms  |  Generation: {generation_ms:.0f} ms  |  Total: {retrieval_ms + generation_ms:.0f} ms')

## Cell 5: Inspect — BM25 vs Dense vs Hybrid Comparison
**What this demonstrates**: Side-by-side retrieval results from all three strategies on the same query, with coverage analysis showing which chunks each modality found and how RRF resolves the difference.
**Expected output**: Three-column comparison table, chunk coverage summary, and RRF score distribution showing which chunks were rewarded for appearing in both lists.

In [ ]:
# ── Run all three retrievers on the same query ────────────────────────────────
bm25_top5  = bm25_retrieve(QUERY, n=5)
dense_top5 = dense_retrieve(QUERY, n=5)
# hybrid_results already computed in Cell 4 (top-5 hybrid)

# ── Three-way side-by-side comparison ────────────────────────────────────────
print(f'Three-way comparison  —  query: {QUERY[:55]!r}...')
print()
hdr_bm25  = 'BM25 only (keyword)'
hdr_dense = 'Dense only (semantic)'
hdr_hyb   = 'Hybrid RRF (fused)'
print(f'  {"Rank":<4}  {hdr_bm25:<34}  {hdr_dense:<34}  {hdr_hyb}')
print('  ' + '-' * 110)

for rank in range(5):
    # BM25 column
    if rank < len(bm25_top5):
        idx, score = bm25_top5[rank]
        bm25_col = f'[{idx:02d}] bm25={score:5.2f}  {chunks[idx][:18].replace(chr(10), " ")}...'
    else:
        bm25_col = '(no result)'
    # Dense column
    if rank < len(dense_top5):
        idx, score = dense_top5[rank]
        dense_col = f'[{idx:02d}] sim={score:.3f}   {chunks[idx][:18].replace(chr(10), " ")}...'
    else:
        dense_col = '(no result)'
    # Hybrid column
    if rank < len(hybrid_results):
        r = hybrid_results[rank]
        src = 'B+D' if r['in_bm25'] and r['in_dense'] else 'B  ' if r['in_bm25'] else '  D'
        hyb_col = f'[{r["chunk_idx"]:02d}] rrf={r["rrf_score"]:.5f} ({src})  {r["text"][:14].replace(chr(10), " ")}...'
    else:
        hyb_col = '(no result)'
    print(f'  {rank+1:<4}  {bm25_col:<34}  {dense_col:<34}  {hyb_col}')

# ── Coverage analysis ─────────────────────────────────────────────────────────
print()
bm25_ids   = {idx for idx, _ in bm25_top5}
dense_ids  = {idx for idx, _ in dense_top5}
hybrid_ids = {r['chunk_idx'] for r in hybrid_results}

in_both       = hybrid_ids & bm25_ids & dense_ids
bm25_only_set = hybrid_ids & bm25_ids - dense_ids
dense_only_set= hybrid_ids & dense_ids - bm25_ids

print('Hybrid coverage breakdown:')
print(f'  Found by BOTH BM25 and Dense:  {len(in_both)} chunk(s)  → highest RRF (rewarded by two lists)')
print(f'  Found by BM25 only:            {len(bm25_only_set)} chunk(s)  → exact keyword hit not in semantic top-5')
print(f'  Found by Dense only:           {len(dense_only_set)} chunk(s)  → semantic match, no exact token overlap')

# ── RRF score distribution ────────────────────────────────────────────────────
print()
print('RRF score distribution (hybrid top-5):')
max_rrf = max(r['rrf_score'] for r in hybrid_results)
for i, r in enumerate(hybrid_results, 1):
    src = 'BM25+Dense' if r['in_bm25'] and r['in_dense'] else 'BM25 only ' if r['in_bm25'] else 'Dense only'
    bar = '█' * int(r['rrf_score'] / max_rrf * 30)   # normalised bar up to 30 chars
    print(f'  [{i}] {r["rrf_score"]:.5f}  ({src})  {bar}')

print()
print('Key observation:')
print('  Chunks appearing in both lists score ~2× a single-list chunk at the same rank.')
print('  This is the RRF reward signal: cross-modal agreement boosts confidence.')
print('  Chunks absent from one list are not penalised — nothing is discarded.')

## Cell 6: Fintech — Close-Out Netting and Early Termination Procedures
**What this demonstrates**: Hybrid RAG applied to a multi-concept ISDA query requiring both exact legal identifiers (Section 6(a), Early Termination Date) and semantic understanding (netting, payment calculation). Shows why neither modality alone is sufficient for derivatives contract review.
**Expected output**: Retriever comparison table, generated answer with Section citations, and an explanation of the hybrid advantage for this query type.

In [ ]:
FINTECH_QUERY = (
    'What are the procedures for close-out netting and calculating termination '
    'payments when an Event of Default occurs under Section 6?'
)

FINTECH_SYSTEM = (
    'You are a derivatives legal counsel assistant specialising in ISDA documentation. '
    'Answer using ONLY the provided ISDA agreement excerpts. '
    'For each procedural step, cite the exact Section reference (e.g. Section 6(a), Section 6(e)(ii)). '
    'Highlight any notice periods, time constraints, or numeric thresholds verbatim. '
    'If the context is insufficient for a complete answer, identify what section type would contain the missing detail.'
)

# ── Run all three retrievers on the fintech query ─────────────────────────────
bm25_ft   = bm25_retrieve(FINTECH_QUERY, n=5)
dense_ft  = dense_retrieve(FINTECH_QUERY, n=5)
hybrid_ft = hybrid_retrieve(FINTECH_QUERY, n=TOP_N, top_k=TOP_K)

# ── Retriever comparison table ─────────────────────────────────────────────────
print(f'Query: {FINTECH_QUERY!r}')
print()
print('Retriever comparison:')
print(f'  {"Rank":<4}  {"BM25 only":<36}  {"Dense only":<36}  Hybrid (RRF)')
print('  ' + '-' * 115)
for rank in range(5):
    bm25_col = dense_col = hyb_col = ''
    if rank < len(bm25_ft):
        idx, score = bm25_ft[rank]
        bm25_col = f'[{idx:02d}] bm25={score:5.2f}  {chunks[idx][:20].replace(chr(10), " ")}...'
    if rank < len(dense_ft):
        idx, score = dense_ft[rank]
        dense_col = f'[{idx:02d}] sim={score:.3f}   {chunks[idx][:20].replace(chr(10), " ")}...'
    if rank < len(hybrid_ft):
        r = hybrid_ft[rank]
        src = 'B+D' if r['in_bm25'] and r['in_dense'] else 'B  ' if r['in_bm25'] else '  D'
        hyb_col = f'[{r["chunk_idx"]:02d}] rrf={r["rrf_score"]:.5f} ({src})'
    print(f'  {rank+1:<4}  {bm25_col:<36}  {dense_col:<36}  {hyb_col}')

# ── Generate the compliance answer ────────────────────────────────────────────
context_ft = '\n\n---\n\n'.join(
    f"[ISDA Excerpt {i+1} | RRF={r['rrf_score']:.5f} | "
    f"{'BM25+Dense' if r['in_bm25'] and r['in_dense'] else 'BM25 only' if r['in_bm25'] else 'Dense only'}]\n"
    f"{r['text']}"
    for i, r in enumerate(hybrid_ft)
)

t_gen = time.perf_counter()
response_ft = client.messages.create(
    model=LLM_MODEL,
    max_tokens=600,
    system=FINTECH_SYSTEM,
    messages=[{
        'role': 'user',
        'content': f'ISDA agreement excerpts:\n{context_ft}\n\nQuestion: {FINTECH_QUERY}'
    }],
)
gen_ms = (time.perf_counter() - t_gen) * 1000
answer_ft: str = response_ft.content[0].text

print()
print('=' * 65)
print('ISDA CONTRACT ANALYSIS — Close-Out Netting Procedures')
print('=' * 65)
print(answer_ft)
print('=' * 65)

# ── Why hybrid matters for this query ─────────────────────────────────────────
print()
print('Why Hybrid RAG matters for this ISDA query:')
print('  BM25 strength: exact identifiers — "Event of Default", "Section 6(a)",')
print('                 "Early Termination Date", "Non-defaulting Party"')
print('                 → tokens that must appear verbatim; semantic drift is a risk')
print('  Dense strength: conceptual context — payment netting methodology, close-out')
print('                  mechanics described in different clause language')
print('                  → Section 6(e) content even when exact phrase is absent')
print('  Hybrid verdict: neither modality alone retrieves the complete answer;')
print('                  RRF fusion ensures no legally significant clause is missed')
print()
print(f'Generation time: {gen_ms:.0f} ms')